In [ ]:
import warnings
import pandas as pd
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

from dataset.configs.feature_configs_general import generate_general_config
from dataset.dukascopy_func import (
    crawl_OHLCV_data_dukascopy
)
from dataset.history_data_stage_one_func import (
    history_data_stage_one
)
from dataset.realtime_candle_func import (
    historiy_realtime_candle
)
from dataset.indicator_func import history_indicator_calculator
from dataset.realtime_shift_func import history_cndl_shift
from dataset.create_basic_features_func import (
    history_basic_features,
    history_fe_market_close,
    history_fe_time,
)
from dataset.window_agg_features_func import (
    history_fe_WIN_features
)
from dataset.columns_merge_func import history_columns_merge
# from dataset.data_crawlers.metatrader_func import (
#     crawl_OHLCV_data_metatrader,
# )
from main_func import main
# from forex_MLOps.utils.wandb_utils import read_obj
from ETL import read_data_manual, ETL
# from forex_MLOps.model_ensemble import QuantEnsemble
from utils.wandb_utils import (
    fetch_artifacts, read_tracker_objects, download_wandb_artifact
)


import polars as pl
########### READ DATA MANUALLY #########
# df_all = read_data_manual()
# !tail -n 120 ./forex_MLOps/ETL.py

In [ ]:
config_general = generate_general_config()

In [ ]:
failed_dates_dict = await crawl_OHLCV_data_dukascopy(
    feature_config = config_general
    
)

In [ ]:
history_data_stage_one(config_general)

In [ ]:
historiy_realtime_candle(config_general)

In [ ]:
history_indicator_calculator(config_general)

In [ ]:
history_cndl_shift(config_general)

In [ ]:
history_basic_features(config_general)
history_fe_market_close(config_general)
history_fe_time(config_general)

In [ ]:
history_fe_WIN_features(config_general)

In [ ]:
history_columns_merge(config_general,general_mode=True)

In [ ]:
model_type = 'XGB' #@param ['XGB', 'RF', 'LGBM'] {type: "string"}
MANUAL_EXP = True #@param {type:"boolean"}
man_params = {'RF':None, 'XGB':None}

In [ ]:
man_params['RF'] = {
    'model_name' : 'RF',

    'target_symbol' : 'EURUSD',
    'trade_mode': 'short' ,   #"long" , "short"
    'trg_look_ahead' : 400,
    'trg_take_profit' : 30,
    'trg_stop_loss' : 11,

    'strg_look_ahead' : 300,
    'strg_take_profit' : 50,
    'strg_stop_loss' : 20,


    'n_rand_features': None,
    'save_model_mode' : None, # None, 'sample_train_size', 'last_train_size', 'all_data',
    'n_splits' : 2,
    'max_train_size' : 900*288,
    'test_size' : 30*288,
    'train_test_gap':30*288,

    'parameters':{
        'n_estimators' : 500,
        'max_depth': 7,
        'max_features' : 128,

        'max_samples' : 0.7,
        'class_weight': {0:1,1:1}, # 'balanced'
        'random_state': 42,
        'n_jobs': -1
        }
}


man_params['XGB'] = {
    'model_name' : 'XGB',

    'target_symbol' : 'EURUSD',
    'trade_mode': 'short' ,   #"long" , "short"
    'trg_look_ahead' : 400,
    'trg_take_profit' : 30,
    'trg_stop_loss' : 11,

    'strg_look_ahead' : 300,
    'strg_take_profit' : 30,
    'strg_stop_loss' : 15,

    'n_rand_features': None,
    'save_model_mode' : None, # None, 'sample_train_size', 'last_train_size', 'all_data',
    'n_splits' : 5,
    'max_train_size' : 900*288,
    'test_size' : 30*288,
    'train_test_gap':30*288,

    "parameters": {
        'tree_method':'hist',#  None,hist
        'device' : 'cpu',#  None,'cuda'
        "objective": "binary:logistic",
        "max_depth": 7,
        "learning_rate": 0.05,
        "n_estimators": 50,
    #         "early_stopping_rounds" :{"distribution": "int_uniform", "min": 5, "max": 150},
        "early_stopping_rounds" :None,

#         "gamma": 7, #? float
#         "min_child_weight": 1,
        # Used to control over-fitting.
#         "max_delta_step": 2, #?float
        "subsample": 0.5,
        "colsample_bytree": 0.8,
        "reg_lambda": 0.2,
        "reg_alpha": 0.15,
        "scale_pos_weight" :1,
        'random_state': 0,
        },
}

In [ ]:
if MANUAL_EXP:
    exp_obj, exp_metadata, artifact_name = main(
        manual = True,
        man_params = man_params[model_type],
        dataset_path = r"C:\Algorithm_Trading\AlphaHunters\ML-Algotrading-Project\dataset\data\dataset\dataset.parquet",
        C5M_data_path = r"C:\Algorithm_Trading\AlphaHunters\ML-Algotrading-Project\dataset\data\stage_one_data",
    )

In [ ]:
imp_fe = list(exp_obj.feature_importance[exp_obj.feature_importance.cv<.5].feature_name)
print(len(exp_obj.feature_importance[exp_obj.feature_importance.cv<0.5].feature_name))
exp_obj.feature_importance[exp_obj.feature_importance.cv<.5].feature_name

In [ ]:
def plot_feature_importances_scatter(importance_df, top_n=50, figsize=(15, 8)):
    # import seaborn
    # Calculate mean importance and sort features
    mean_importance = importance_df['mean_importance']
    sorted_idx = mean_importance.argsort()[::-1]
    top_features = importance_df.iloc[sorted_idx[:top_n]]
    imp_cols = [f for f in importance_df if 'importance_fold' in f]
    # Prepare data for scatter plot
    feature_names = top_features['feature_name']
    importance_values = top_features[imp_cols].values.T  # Transpose for plotting

    # Calculate median importance
    median_importance = top_features['median_importance']
    # print(median_importance)
    # Create scatter plot
    fig, ax = plt.subplots(figsize=figsize)

    for i, feature in enumerate(feature_names):
        ax.scatter([i] * importance_values.shape[0], importance_values[:, i],
                   alpha=0.6, label=feature)

    # Plot median line
    ax.plot(range(top_n), median_importance, color='red', linestyle='--',
            linewidth=2, label='Median')

    # Add value labels on the median line
    for i, v in enumerate(median_importance):
        ax.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

    ax.set_xlabel('Features', fontsize=12)
    ax.set_ylabel('Feature Importance', fontsize=12)
    ax.set_title(f'Top {top_n} Feature Importances Across Folds with Median', fontsize=14)
    ax.set_xticks(range(top_n))
    ax.set_xticklabels(feature_names, rotation=45, ha='right')
#     ax.legend(title='Features', bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    plt.show()


import matplotlib.pyplot as plt
# import seaborn as sns
plot_feature_importances_scatter(exp_obj.feature_importance)

In [ ]:

df_fea_imp_sorted = exp_obj.feature_importance
# def MI_WM (name_feature1 , name_feature2 , name_feature3):
#     flag = []
#     for name_feature in [name_feature1 , name_feature2 , name_feature3]:
#         flag.append(df_fea_imp_sorted[df_fea_imp_sorted['feature_name'] == name_feature].mean_importance.values[0])
#     return max(flag)
# print(MI_WM('RANDOM_0' , 'RANDOM_1' , 'RANDOM_2'))
feature_0 = df_fea_imp_sorted[df_fea_imp_sorted['mean_importance']==0].feature_name
feature_0_unique = set([f.split('_')[1] for f in feature_0 if len(f.split('_'))>1 ])
dict_feature0 = {
    'GMA':{'tedad':0,'name':[]},
    'RSI':{'tedad':0,'name':[]},
    'WIN':{'tedad':0,'name':[]},
    'cndl':{'tedad':0,'name':[]},
    'ratio':{'tedad':0,'name':[]},
    'shape':{'tedad':0,'name':[]},
    'time':{'tedad':0,'name':[]}, 
}
for f_name in feature_0:
    if f_name.split('_')[1] in dict_feature0.keys():
        dict_feature0[f_name.split('_')[1]]['tedad'] +=1
        dict_feature0[f_name.split('_')[1]]['name'].append(f_name)
    else:
        dict_feature0.update({f_name.split('_')[1]:{'tedad':1,'name':[f_name]}})

print(f'Tedad Feature0 : {len(feature_0)}')
dict_feature0